# Experimentation Notebook

This notebook implements the experimentation workflow described in `experimentation-plan.md`.

It is designed to:
- load the synthesized French-Swahili-Lingala text and audio artifacts,
- verify official splits and basic dataset integrity,
- construct controlled retrieval benchmarks,
- evaluate retrieval with reproducible metrics,
- run statistical significance tests,
- support structured error analysis,
- keep conclusions scoped to the datasets in this workspace.

The notebook starts with a lightweight lexical baseline that should run locally. Optional model hooks for SERENGETI, AfriBERTa, and CLAP-style experiments are included as scaffolding for later runs.

In [1]:
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Optional, Sequence

import json
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize

RNG = np.random.default_rng(42)
random.seed(42)

ROOT = Path.cwd()
TEXT_PATH = ROOT / "synthesis_outputs" / "text" / "cs_text_dataset.jsonl"
AUDIO_MANIFEST_PATH = ROOT / "synthesis_outputs" / "manifests" / "audio_manifest.jsonl"
SPLITS_PATH = ROOT / "synthesis_outputs" / "manifests" / "splits.json"
SUMMARY_PATH = ROOT / "synthesis_outputs" / "manifests" / "summary.json"
DATACARD_PATH = ROOT / "synthesis_outputs" / "manifests" / "DATACARD.json"
EXPERIMENT_OUTPUT_DIR = ROOT / "experiment_outputs"
CSV_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / "csv"
PLOT_OUTPUT_DIR = EXPERIMENT_OUTPUT_DIR / "plots"

CSV_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
PLOT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TEXT_PATH, AUDIO_MANIFEST_PATH, SPLITS_PATH

(WindowsPath('c:/cmu/course-work/spring-1/applications-ai-africa/group-work/synthesis_outputs/text/cs_text_dataset.jsonl'),
 WindowsPath('c:/cmu/course-work/spring-1/applications-ai-africa/group-work/synthesis_outputs/manifests/audio_manifest.jsonl'),
 WindowsPath('c:/cmu/course-work/spring-1/applications-ai-africa/group-work/synthesis_outputs/manifests/splits.json'))

In [2]:
def load_json(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def load_jsonl(path: Path) -> List[dict]:
    records: List[dict] = []
    with path.open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records


def extract_source_family(source_trace: dict) -> str:
    if not isinstance(source_trace, dict):
        return "unknown"
    for key in ("source_id", "parent", "sw_sentence_id", "fr_sentence_id", "text_id"):
        value = source_trace.get(key)
        if value:
            return str(value)
    return "unknown"


def switch_ratio(lang_tags: Sequence[str]) -> float:
    valid = [tag for tag in lang_tags if tag not in {"punct", "other", None}]
    if not valid:
        return 0.0
    matrix_lang = max(set(valid), key=valid.count)
    switched = sum(1 for tag in valid if tag != matrix_lang)
    return switched / len(valid)


def switch_band(value: float) -> str:
    if value < 0.2:
        return "low"
    if value < 0.5:
        return "medium"
    return "high"


text_records = load_jsonl(TEXT_PATH)
audio_records = load_jsonl(AUDIO_MANIFEST_PATH)
splits = load_json(SPLITS_PATH)
summary = load_json(SUMMARY_PATH)
datacard = load_json(DATACARD_PATH)

text_df = pd.DataFrame(text_records)
text_df["source_family"] = text_df["source_trace"].apply(extract_source_family)
text_df["switch_ratio"] = text_df["lang_tags"].apply(switch_ratio)
text_df["switch_band"] = text_df["switch_ratio"].apply(switch_band)
text_df["has_lingala"] = text_df["lang_tags"].apply(lambda tags: "lin" in tags)
text_df["n_switch_points"] = text_df["switch_points"].apply(lambda value: len(value) if isinstance(value, list) else 0)
text_df["split"] = "unassigned"

for split_name, ids in splits["text"].items():
    text_df.loc[text_df["id"].isin(ids), "split"] = split_name


audio_df = pd.DataFrame(audio_records)
if not audio_df.empty:
    audio_df["text_id"] = audio_df["synthesis_trace"].apply(lambda trace: trace.get("text_id") if isinstance(trace, dict) else None)
    audio_df["split"] = "unassigned"
    for split_name, ids in splits["audio"].items():
        audio_df.loc[audio_df["id"].isin(ids), "split"] = split_name

print(f"Loaded {len(text_df):,} text records")
print(f"Loaded {len(audio_df):,} audio manifest records")

Loaded 9,000 text records
Loaded 623 audio manifest records


## Dataset Integrity and Scope Checks

These checks make the benchmark assumptions explicit before any model evaluation.

They verify:
- official text and audio split sizes,
- distribution of switch intensity,
- whether Lingala-bearing examples are present,
- whether source-lineage leakage is visible across splits,
- whether the authoritative audio manifest is internally consistent.

If these checks fail, results should not be trusted.

In [3]:
def leakage_report(frame: pd.DataFrame, split_col: str = "split") -> pd.DataFrame:
    grouped = frame.groupby("source_family")[split_col].nunique()
    leak_sources = grouped[grouped > 1]
    return leak_sources.rename("n_splits").reset_index().sort_values("n_splits", ascending=False)


text_summary = pd.DataFrame(
    {
        "count": text_df.groupby("split")["id"].count(),
        "avg_switch_ratio": text_df.groupby("split")["switch_ratio"].mean().round(3),
        "lingala_rate": text_df.groupby("split")["has_lingala"].mean().round(3),
        "avg_switch_points": text_df.groupby("split")["n_switch_points"].mean().round(2),
    }
).reset_index()

print("Text split summary")
display(text_summary)

print("Switch-band distribution on text test split")
switch_band_distribution = (
    text_df.query("split == 'test'")
    .groupby("switch_band")["id"]
    .count()
    .rename("count")
    .reset_index()
)
display(switch_band_distribution)

text_leaks = leakage_report(text_df)
print(f"Potential text source-family leakage count: {len(text_leaks):,}")
if len(text_leaks):
    display(text_leaks.head(10))

if not audio_df.empty:
    print("Audio split summary")
    audio_split_summary = audio_df.groupby("split")["id"].count().rename("count").reset_index()
    display(audio_split_summary)

print("Manifest summary file")
manifest_summary_df = pd.json_normalize(summary)
display(manifest_summary_df)

if not audio_df.empty:
    audio_qc_summary = audio_df["quality"].apply(pd.Series)
    print("Audio quality summary from the authoritative manifest")
    audio_quality_table = pd.DataFrame(
        {
            "qc_pass_rate": [audio_qc_summary["qc_pass"].mean()],
            "avg_mos_proxy": [audio_qc_summary["mos_proxy"].mean()],
            "avg_clipping": [audio_qc_summary["clipping_ratio"].mean()],
            "avg_asr_backtrans_wer": [audio_qc_summary["asr_backtrans_wer"].mean()],
        }
    ).round(3)
    display(audio_quality_table)

Text split summary


,split,count,avg_switch_ratio,lingala_rate,avg_switch_points
0,dev,900,0.041,0.176,0.41
1,test,900,0.046,0.189,0.46
2,train,7200,0.042,0.171,0.44


Switch-band distribution on text test split


,switch_band,count
0,high,12
1,low,846
2,medium,42


Potential text source-family leakage count: 1,472


...

,split,count
0,dev,62
1,test,63
2,train,498


Manifest summary file


,date,text_records,audio_records,anti_leakage_ok,text_split.train,text_split.dev,text_split.test,audio_split.train,audio_split.dev,audio_split.test,kpi.avg_cs_acceptability,kpi.avg_mos_proxy,kpi.avg_asr_backtrans_wer
0,2026-04-13,9000,623,True,7200,900,900,498,62,63,0.757,4.4,0.12


Audio quality summary from the authoritative manifest


,qc_pass_rate,avg_mos_proxy,avg_clipping,avg_asr_backtrans_wer
0,1.0,4.4,0.0,0.12


## Text Retrieval Benchmark Construction

The first experiment is text-to-text retrieval because it isolates code-switch retrieval behavior before adding audio uncertainty.

The benchmark below uses the official text split and constructs a controlled candidate pool per query:
- 1 gold target,
- 9 hard negatives chosen by lexical similarity,
- 40 random negatives.

Gold items are selected by matching source lineage where possible while avoiding self-retrieval.

In [4]:
def build_text_gold_map(test_frame: pd.DataFrame) -> Dict[str, str]:
    by_source = test_frame.groupby("source_family")["id"].apply(list).to_dict()
    gold_map: Dict[str, str] = {}
    for ids in by_source.values():
        if len(ids) < 2:
            continue
        for index, query_id in enumerate(ids):
            gold_map[query_id] = ids[(index + 1) % len(ids)]
    return gold_map


def build_candidate_pool(
    query_id: str,
    gold_id: str,
    similarity_frame: pd.DataFrame,
    test_ids: Sequence[str],
    hard_negative_k: int = 9,
    random_negative_k: int = 40,
) -> List[str]:
    ranking = similarity_frame.loc[query_id].sort_values(ascending=False).index.tolist()
    negatives = [candidate for candidate in ranking if candidate not in {query_id, gold_id}]
    hard_negatives = negatives[:hard_negative_k]
    remaining = [candidate for candidate in test_ids if candidate not in {query_id, gold_id, *hard_negatives}]
    sample_size = min(random_negative_k, len(remaining))
    random_negatives = RNG.choice(np.array(remaining, dtype=object), size=sample_size, replace=False).tolist() if sample_size else []
    pool = [gold_id, *hard_negatives, *random_negatives]
    return pool


test_text_df = text_df.query("split == 'test'").copy()
gold_map = build_text_gold_map(test_text_df)
eligible_test_df = test_text_df[test_text_df["id"].isin(gold_map.keys())].copy()

baseline_vectorizer = TfidfVectorizer(analyzer="word", ngram_range=(1, 2), min_df=1)
test_matrix = baseline_vectorizer.fit_transform(eligible_test_df["text"])
similarity = cosine_similarity(test_matrix)
similarity_df = pd.DataFrame(similarity, index=eligible_test_df["id"], columns=eligible_test_df["id"])

candidate_pools = {}
for query_id, gold_id in gold_map.items():
    if query_id in similarity_df.index and gold_id in similarity_df.columns:
        candidate_pools[query_id] = build_candidate_pool(
            query_id=query_id,
            gold_id=gold_id,
            similarity_frame=similarity_df,
            test_ids=eligible_test_df["id"].tolist(),
        )

benchmark_df = eligible_test_df[["id", "text", "switch_ratio", "switch_band", "has_lingala", "matrix_lang", "n_switch_points"]].copy()
benchmark_df["gold_id"] = benchmark_df["id"].map(gold_map)
benchmark_df["candidate_pool_size"] = benchmark_df["id"].map(lambda value: len(candidate_pools.get(value, [])))

print(f"Eligible text queries with gold targets: {len(benchmark_df):,}")
display(benchmark_df.head())

print("Candidate pool size distribution")
display(benchmark_df["candidate_pool_size"].value_counts().sort_index().rename_axis("pool_size").reset_index(name="count"))

Eligible text queries with gold targets: 164


,id,text,switch_ratio,switch_band,has_lingala,matrix_lang,n_switch_points,gold_id,candidate_pool_size
21,cs_txt_000022,Wananchi wanataka usalama na barabara routes.,0.0,low,False,swa,0,cs_txt_000023,50
22,cs_txt_000023,Wananchi wanataka usalama na barabarae routes.,0.0,low,False,swa,0,cs_txt_000022,50
39,cs_txt_000040,Mkutano wa hadhara ulifanyika jana soir.,0.0,low,False,swa,0,cs_txt_000041,50
40,cs_txt_000041,Mkutanoe wa hadhara ulifanyika jana soir.,0.0,low,False,swa,0,cs_txt_000040,50
276,cs_txt_000277,Wananchi wanataka securite na barabara bora.,0.0,low,False,swa,0,cs_txt_000279,50


Candidate pool size distribution


,pool_size,count
0,50,164


## Current Benchmark Caveats

The strict T2T benchmark in this notebook defines gold targets by same-split source lineage while avoiding self-retrieval.

That choice is conservative, but it means the current automatic benchmark covers only the subset of test items that have a same-lineage partner inside the test split. In the present artifacts, that yields fewer than the full 900 text test queries.

Interpretation:
- the current T2T benchmark is valid for controlled comparison,
- it is not yet the final 900-query benchmark envisioned in the plan,
- expanding to 900 queries will require either manual relevance annotation or a stronger gold-pair construction procedure.

Audio analysis in this notebook now uses only `audio_manifest.jsonl` as the authoritative manifest.

## Metrics and Statistical Testing

This section implements the primary metrics from the experimentation plan:
- MRR
- Recall@1
- nDCG@5
- CSRS

It also adds paired significance tests:
- paired bootstrap for metric differences,
- McNemar's test for top-1 success differences.

In [5]:
def reciprocal_rank(ranked_ids: Sequence[str], gold_id: str) -> float:
    for rank, candidate_id in enumerate(ranked_ids, start=1):
        if candidate_id == gold_id:
            return 1.0 / rank
    return 0.0


def recall_at_k(ranked_ids: Sequence[str], gold_id: str, k: int) -> float:
    return float(gold_id in ranked_ids[:k])


def ndcg_at_k(ranked_ids: Sequence[str], gold_id: str, k: int) -> float:
    for rank, candidate_id in enumerate(ranked_ids[:k], start=1):
        if candidate_id == gold_id:
            return 1.0 / math.log2(rank + 1)
    return 0.0


def holm_bonferroni(p_values: Sequence[float]) -> List[float]:
    indexed = sorted(enumerate(p_values), key=lambda item: item[1])
    corrected = [0.0] * len(p_values)
    total = len(p_values)
    running_max = 0.0
    for order, (original_index, p_value) in enumerate(indexed, start=1):
        adjusted = min(1.0, (total - order + 1) * p_value)
        running_max = max(running_max, adjusted)
        corrected[original_index] = running_max
    return corrected


def paired_bootstrap_diff(
    values_a: np.ndarray,
    values_b: np.ndarray,
    n_resamples: int = 10_000,
    seed: int = 42,
) -> dict:
    rng = np.random.default_rng(seed)
    n_items = len(values_a)
    diffs = np.empty(n_resamples, dtype=float)
    indices = np.arange(n_items)
    for idx in range(n_resamples):
        sample = rng.choice(indices, size=n_items, replace=True)
        diffs[idx] = values_a[sample].mean() - values_b[sample].mean()
    observed = values_a.mean() - values_b.mean()
    ci_low, ci_high = np.quantile(diffs, [0.025, 0.975])
    two_sided_p = 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())
    return {
        "observed_diff": observed,
        "ci_low": ci_low,
        "ci_high": ci_high,
        "p_value": min(1.0, float(two_sided_p)),
    }


def mcnemar_test(success_a: np.ndarray, success_b: np.ndarray) -> dict:
    b = int(np.sum((success_a == 1) & (success_b == 0)))
    c = int(np.sum((success_a == 0) & (success_b == 1)))
    n = b + c
    p_value = 1.0 if n == 0 else stats.binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue
    return {"b": b, "c": c, "p_value": float(p_value)}


def summarise_rank_metrics(results: pd.DataFrame) -> dict:
    metrics = {
        "MRR": results["rr"].mean(),
        "Recall@1": results["r1"].mean(),
        "nDCG@5": results["ndcg5"].mean(),
    }
    band_means = results.groupby("switch_band")["rr"].mean().to_dict()
    low = band_means.get("low", np.nan)
    high = band_means.get("high", np.nan)
    if np.isnan(low) or np.isnan(high) or low == 0:
        metrics["CSRS"] = np.nan
    else:
        metrics["CSRS"] = float(high / low)
    return metrics

## Baseline Retrieval Experiment

This baseline is intentionally lightweight. It uses TF-IDF retrieval over the benchmark candidate pools.

It is not meant to be competitive with SERENGETI or AfriBERTa. Its role is to:
- verify that the benchmark pipeline works,
- establish a reproducible reference point,
- produce per-query outputs for later significance testing and error analysis.

In [6]:
def rank_candidates_with_similarity(
    query_id: str,
    candidate_ids: Sequence[str],
    similarity_frame: pd.DataFrame,
) -> List[str]:
    scores = similarity_frame.loc[query_id, list(candidate_ids)]
    return scores.sort_values(ascending=False).index.tolist()


def run_t2t_similarity_experiment(
    benchmark_frame: pd.DataFrame,
    candidate_pools: Dict[str, List[str]],
    similarity_frame: pd.DataFrame,
    label: str,
) -> pd.DataFrame:
    rows = []
    for row in benchmark_frame.itertuples(index=False):
        pool = candidate_pools.get(row.id, [])
        if not pool:
            continue
        ranked = rank_candidates_with_similarity(row.id, pool, similarity_frame)
        rows.append(
            {
                "model": label,
                "query_id": row.id,
                "gold_id": row.gold_id,
                "ranked_ids": ranked,
                "rr": reciprocal_rank(ranked, row.gold_id),
                "r1": recall_at_k(ranked, row.gold_id, 1),
                "ndcg5": ndcg_at_k(ranked, row.gold_id, 5),
                "switch_band": row.switch_band,
                "switch_ratio": row.switch_ratio,
                "has_lingala": row.has_lingala,
                "matrix_lang": row.matrix_lang,
                "n_switch_points": row.n_switch_points,
            }
        )
    return pd.DataFrame(rows)


baseline_results = run_t2t_similarity_experiment(
    benchmark_frame=benchmark_df,
    candidate_pools=candidate_pools,
    similarity_frame=similarity_df,
    label="tfidf_word_bigram",
)

baseline_summary = pd.DataFrame([summarise_rank_metrics(baseline_results)]).round(4)
print("Baseline T2T metrics")
display(baseline_summary)

print("Baseline T2T by switch band")
display(
    baseline_results.groupby("switch_band")[["rr", "r1", "ndcg5"]].mean().round(4).reset_index()
)

print("Baseline T2T by Lingala presence")
display(
    baseline_results.groupby("has_lingala")[["rr", "r1", "ndcg5"]].mean().round(4).reset_index()
)

Baseline T2T metrics


,MRR,Recall@1,nDCG@5,CSRS
0,0.5803,0.4085,0.6232,1.4277


Baseline T2T by switch band


,switch_band,rr,r1,ndcg5
0,high,0.8333,0.6667,0.8770
1,low,0.5837,0.4156,0.6238
2,medium,0.3976,0.1429,0.5025


Baseline T2T by Lingala presence


,has_lingala,rr,r1,ndcg5
0,False,0.5912,0.4154,0.6309
1,True,0.5385,0.3824,0.5938


## Paired Comparison Template

Use this section after adding a second model's per-query results. The same code can compare:
- lexical baseline vs SERENGETI,
- SERENGETI baseline vs SERENGETI plus synthetic CS training,
- AfriBERTa baseline vs AfriBERTa plus synthetic CS training.

The comparison is query-paired by construction.

In [7]:
def compare_result_frames(frame_a: pd.DataFrame, frame_b: pd.DataFrame, label_a: str, label_b: str) -> pd.DataFrame:
    joined = frame_a[["query_id", "rr", "r1", "ndcg5"]].merge(
        frame_b[["query_id", "rr", "r1", "ndcg5"]],
        on="query_id",
        suffixes=("_a", "_b"),
    )

    rr_boot = paired_bootstrap_diff(joined["rr_b"].to_numpy(), joined["rr_a"].to_numpy())
    ndcg_boot = paired_bootstrap_diff(joined["ndcg5_b"].to_numpy(), joined["ndcg5_a"].to_numpy())
    mcnemar = mcnemar_test(joined["r1_a"].to_numpy(), joined["r1_b"].to_numpy())

    return pd.DataFrame(
        [
            {
                "comparison": f"{label_b} vs {label_a}",
                "metric": "MRR",
                "delta": rr_boot["observed_diff"],
                "ci_low": rr_boot["ci_low"],
                "ci_high": rr_boot["ci_high"],
                "p_value": rr_boot["p_value"],
            },
            {
                "comparison": f"{label_b} vs {label_a}",
                "metric": "nDCG@5",
                "delta": ndcg_boot["observed_diff"],
                "ci_low": ndcg_boot["ci_low"],
                "ci_high": ndcg_boot["ci_high"],
                "p_value": ndcg_boot["p_value"],
            },
            {
                "comparison": f"{label_b} vs {label_a}",
                "metric": "Recall@1",
                "delta": joined["r1_b"].mean() - joined["r1_a"].mean(),
                "ci_low": np.nan,
                "ci_high": np.nan,
                "p_value": mcnemar["p_value"],
            },
        ]
    )


baseline_vs_self = compare_result_frames(
    baseline_results,
    baseline_results,
    label_a="tfidf_word_bigram",
    label_b="tfidf_word_bigram",
).round(4)

print("Sanity check: self-comparison should show zero deltas")
display(baseline_vs_self)

Sanity check: self-comparison should show zero deltas


,comparison,metric,delta,ci_low,ci_high,p_value
0,tfidf_word_bigram vs tfidf_word_bigram,MRR,0.0,0.0,0.0,1.0
1,tfidf_word_bigram vs tfidf_word_bigram,nDCG@5,0.0,0.0,0.0,1.0
2,tfidf_word_bigram vs tfidf_word_bigram,Recall@1,0.0,NaN,NaN,1.0


## Error Analysis Workspace

This section surfaces failed queries and organizes them for manual inspection.

It supports the error categories from the plan:
- semantic confusion,
- language-identity bias,
- switch-boundary failure,
- Lingala under-representation,
- named-entity failure,
- synthetic artifact sensitivity,
- audio boundary failure.

The notebook cannot infer those labels perfectly on its own, but it can prepare the examples that need review.

In [8]:
def build_error_analysis_frame(results: pd.DataFrame, text_lookup: pd.DataFrame, limit: int = 30) -> pd.DataFrame:
    lookup = text_lookup.set_index("id")
    failures = results[results["r1"] == 0].copy()
    failures = failures.sort_values(["switch_band", "has_lingala", "rr"], ascending=[True, False, True]).head(limit)

    rows = []
    for row in failures.itertuples(index=False):
        ranked_ids = row.ranked_ids
        predicted_id = ranked_ids[0] if ranked_ids else None
        gold_text = lookup.loc[row.gold_id, "text"] if row.gold_id in lookup.index else None
        pred_text = lookup.loc[predicted_id, "text"] if predicted_id in lookup.index else None
        rows.append(
            {
                "query_id": row.query_id,
                "query_text": lookup.loc[row.query_id, "text"],
                "gold_id": row.gold_id,
                "gold_text": gold_text,
                "predicted_id": predicted_id,
                "predicted_text": pred_text,
                "switch_band": row.switch_band,
                "has_lingala": row.has_lingala,
                "n_switch_points": row.n_switch_points,
                "proposed_error_type": "",
                "likely_cause": "",
                "notes": "",
            }
        )
    return pd.DataFrame(rows)


error_analysis_df = build_error_analysis_frame(baseline_results, eligible_test_df, limit=30)
display(error_analysis_df.head(10))

,query_id,query_text,gold_id,gold_text,predicted_id,predicted_text,switch_band,has_lingala,n_switch_points,proposed_error_type,likely_cause,notes
0,cs_txt_001453,Bajeti ya batu imeongezeka augmente huu.,cs_txt_001454,Bajeti ya batu ndenge augmente huu.,cs_txt_000523,Bajeti ya afya imeongezeka augmente huu.,high,True,4,,,
1,cs_txt_005608,Bajeti ya afya imeongezeka mwaka annee.,cs_txt_005610,Bajeti ya afyae imeongezeka mwaka anneee.,cs_txt_008818,Bajeti ya afya imeongezeka mwaka annee.,low,True,2,,,
2,cs_txt_003163,Bajeti ya afya imeongezeka cette huu.,cs_txt_003165,Bajetie ya afya imeongezeka batu huu.,cs_txt_000523,Bajeti ya afya imeongezeka augmente huu.,low,True,2,,,
3,cs_txt_008323,Bajeti ya afya imeongezeka mwaka annee.,cs_txt_008325,Bajeti ya afya imeongezeka mwakae annee.,cs_txt_008818,Bajeti ya afya imeongezeka mwaka annee.,low,True,2,,,
4,cs_txt_000868,Bajeti ya afya imeongezeka mwaka annee.,cs_txt_000869,Bajeti ya afya imeongezeka mwaka anneee.,cs_txt_005608,Bajeti ya afya imeongezeka mwaka annee.,low,True,2,,,
5,cs_txt_008818,Bajeti ya afya imeongezeka mwaka annee.,cs_txt_008820,Bajeti ya afya imeongezeka mwaka ndenge.,cs_txt_000868,Bajeti ya afya imeongezeka mwaka annee.,low,True,2,,,
6,cs_txt_006511,Leo tunaongea kuhusu services za ya mjini.,cs_txt_006513,Leo tunaongeae kuhusu servicese za ya mjini.,cs_txt_004741,Leo tunaongea kuhusu services za maji mjini.,low,True,2,,,
7,cs_txt_002047,Wananchi wanataka soki na meilleures bora.,cs_txt_002049,Wananchi wanataka soki batu meilleures bora.,cs_txt_004747,Wananchi wanataka usalama na meilleures bora.,low,True,2,,,
8,cs_txt_003925,Mkutano ya hadhara lieu jana jioni.,cs_txt_003927,Mkutano ya ndenge lieu jana jioni.,cs_txt_002980,Mkutano wa hadhara lieu jana jioni.,low,True,2,,,
9,cs_txt_004282,Wananchi wanataka soki na barabara routes.,cs_txt_004283,Wananchi wanatakae soki na barabara routes.,cs_txt_006757,Wananchi wanataka soki na barabara bora.,low,True,2,,,


In [9]:
baseline_by_switch_band = baseline_results.groupby("switch_band")[["rr", "r1", "ndcg5"]].mean().round(4).reset_index()
baseline_by_lingala = baseline_results.groupby("has_lingala")[["rr", "r1", "ndcg5"]].mean().round(4).reset_index()
candidate_pool_distribution = benchmark_df["candidate_pool_size"].value_counts().sort_index().rename_axis("pool_size").reset_index(name="count")

export_tables = {
    "text_split_summary.csv": text_summary,
    "text_switch_band_distribution.csv": switch_band_distribution,
    "text_leakage_report.csv": text_leaks,
    "audio_split_summary.csv": audio_split_summary if not audio_df.empty else pd.DataFrame(),
    "audio_quality_summary.csv": audio_quality_table if not audio_df.empty else pd.DataFrame(),
    "manifest_summary.csv": manifest_summary_df,
    "benchmark_queries.csv": benchmark_df,
    "candidate_pool_distribution.csv": candidate_pool_distribution,
    "baseline_summary.csv": baseline_summary,
    "baseline_by_switch_band.csv": baseline_by_switch_band,
    "baseline_by_lingala.csv": baseline_by_lingala,
    "baseline_vs_self.csv": baseline_vs_self,
    "baseline_results_per_query.csv": baseline_results.drop(columns=["ranked_ids"]),
    "error_analysis_template.csv": error_analysis_df,
}

for filename, frame in export_tables.items():
    frame.to_csv(CSV_OUTPUT_DIR / filename, index=False)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(baseline_by_switch_band["switch_band"], baseline_by_switch_band["rr"], color=["#4C78A8", "#F58518", "#54A24B"])
ax.set_title("Baseline MRR by Switch Band")
ax.set_xlabel("Switch band")
ax.set_ylabel("MRR")
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(PLOT_OUTPUT_DIR / "baseline_mrr_by_switch_band.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.5))
lingala_labels = baseline_by_lingala["has_lingala"].map({False: "No Lingala", True: "Has Lingala"})
ax.bar(lingala_labels, baseline_by_lingala["rr"], color=["#72B7B2", "#E45756"])
ax.set_title("Baseline MRR by Lingala Presence")
ax.set_xlabel("Subset")
ax.set_ylabel("MRR")
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(PLOT_OUTPUT_DIR / "baseline_mrr_by_lingala_presence.png", dpi=160, bbox_inches="tight")
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(candidate_pool_distribution["pool_size"].astype(str), candidate_pool_distribution["count"], color="#B279A2")
ax.set_title("Candidate Pool Size Distribution")
ax.set_xlabel("Pool size")
ax.set_ylabel("Query count")
fig.tight_layout()
fig.savefig(PLOT_OUTPUT_DIR / "candidate_pool_size_distribution.png", dpi=160, bbox_inches="tight")
plt.close(fig)

print(f"Saved CSV outputs to {CSV_OUTPUT_DIR}")
print(f"Saved plot outputs to {PLOT_OUTPUT_DIR}")
display(pd.DataFrame({
    "csv_files": sorted(path.name for path in CSV_OUTPUT_DIR.glob("*.csv")),
}))
display(pd.DataFrame({
    "plot_files": sorted(path.name for path in PLOT_OUTPUT_DIR.glob("*.png")),
}))

Saved CSV outputs to c:\cmu\course-work\spring-1\applications-ai-africa\group-work\experiment_outputs\csv
Saved plot outputs to c:\cmu\course-work\spring-1\applications-ai-africa\group-work\experiment_outputs\plots


,csv_files
0,audio_quality_summary.csv
1,audio_split_summary.csv
2,baseline_by_lingala.csv
3,baseline_by_switch_band.csv
4,baseline_results_per_query.csv
5,baseline_summary.csv
6,baseline_vs_self.csv
7,benchmark_queries.csv
8,candidate_pool_distribution.csv
9,error_analysis_template.csv


,plot_files
0,baseline_mrr_by_lingala_presence.png
1,baseline_mrr_by_switch_band.png
2,candidate_pool_size_distribution.png


## Persist Tables and Plots

This section saves the main experiment artifacts so they can be used outside the notebook.

Saved outputs include:
- CSV tables for split summaries, benchmark definitions, baseline metrics, significance tables, and error-analysis sheets,
- plot files for switch-band performance, Lingala-stratified performance, and candidate-pool diagnostics.

All files are written under `experiment_outputs/`.

## Optional Transformer Encoder Hooks

These hooks are intentionally off by default. They are here so the notebook can be extended to the planned model comparisons without restructuring the workflow.

Use them only after confirming model availability and compute budget.

In [13]:
import os

RUN_HEAVY_MODELS = os.environ.get("RUN_HEAVY_MODELS", "0") == "1"

if RUN_HEAVY_MODELS:
    from transformers import AutoModel, AutoTokenizer
    import torch

    class MeanPoolingEncoder:
        def __init__(self, model_name: str, device: Optional[str] = None):
            self.model_name = model_name
            self.tokenizer = AutoTokenizer.from_pretrained(model_name)
            self.model = AutoModel.from_pretrained(model_name)
            self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
            self.model.to(self.device)
            self.model.eval()

        def encode(self, texts: Sequence[str], batch_size: int = 16) -> np.ndarray:
            embeddings: List[np.ndarray] = []
            for start in range(0, len(texts), batch_size):
                batch = list(texts[start : start + batch_size])
                encoded = self.tokenizer(batch, padding=True, truncation=True, return_tensors="pt")
                encoded = {key: value.to(self.device) for key, value in encoded.items()}
                with torch.no_grad():
                    outputs = self.model(**encoded)
                hidden = outputs.last_hidden_state
                mask = encoded["attention_mask"].unsqueeze(-1)
                pooled = (hidden * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                embeddings.append(pooled.detach().cpu().numpy())
            return normalize(np.vstack(embeddings))

    # Example placeholders. Replace with exact checked model identifiers before running.
    # serengeti_encoder = MeanPoolingEncoder("UBC-NLP/serengeti")
    # afriberta_encoder = MeanPoolingEncoder("castorini/afriberta_large")
    print("Heavy model hooks enabled. Instantiate encoders explicitly before running experiments.")
else:
    print("Heavy model hooks are disabled. Set environment variable RUN_HEAVY_MODELS=1 after confirming model availability.")

Heavy model hooks are disabled. Set environment variable RUN_HEAVY_MODELS=1 after confirming model availability.


## Conclusion Checklist

Before writing the final report, verify that the experiment output includes:
- exact split and manifest used,
- model and training condition,
- MRR, Recall@1, nDCG@5, and CSRS where applicable,
- confidence intervals,
- significance tests,
- representative failure examples,
- conclusions limited to these datasets.

If the audio analysis uses the v2 manifest, label the results as provisional unless the QC inconsistency is resolved.